## DATA CLEANING


In [3]:
from utils import load_data
df = load_data()
df.shape

(91250, 15)

In [4]:
print("rows, cols:", df.shape)
print("nulls:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())
print("key unique:", not df.duplicated(["Date", "SKU_ID", "Warehouse_ID"]).any())
print("negative values:", (df.select_dtypes("number") < 0).sum().sum())
print("price below cost:", (df["Unit_Price"] < df["Unit_Cost"]).sum())
print("Stockout_Flag values:", df["Stockout_Flag"].unique())
print("first and last date:", df["Date"].min().date(), df["Date"].max().date())

rows, cols: (91250, 15)
nulls: 0
duplicate rows: 0
key unique: True
negative values: 0
price below cost: 0
Stockout_Flag values: [0]
first and last date: 2024-01-01 2024-12-30


## Verifying the primary key

**Claim:** the combination (`Date`, `SKU_ID`, `Warehouse_ID`) uniquely identifies every row, so it can be the primary key.

**Two conditions to prove:**
1. **Unique:** no two rows share the same Date + SKU_ID + Warehouse_ID.
2. **Minimal:** removing any one of the three columns makes the combination non-unique, so all three are needed.

**Why we expect this:** there are 365 days x 50 products x 5 warehouses = 91,250 combinations, which equals the number of rows.

The cell below tests both conditions.

In [5]:
cols = ["Date", "SKU_ID", "Warehouse_ID"]

# 1. The full combination should be unique (no duplicate rows)
print("Date+SKU+Warehouse unique:", not df.duplicated(cols).any())

# 2. Every smaller combination should FAIL to be unique
for drop in cols:
    smaller = [c for c in cols if c != drop]
    print(f"Without {drop:13s} unique?", not df.duplicated(smaller).any())

# 3. Counts that explain why
print("distinct dates:", df.Date.nunique(), "| SKUs:", df.SKU_ID.nunique(), "| warehouses:", df.Warehouse_ID.nunique())
print("365 x 50 x 5 =", 365 * 50 * 5, "| rows:", len(df))

Date+SKU+Warehouse unique: True
Without Date          unique? False
Without SKU_ID        unique? False
Without Warehouse_ID  unique? False
distinct dates: 365 | SKUs: 50 | warehouses: 5
365 x 50 x 5 = 91250 | rows: 91250


In [6]:
fixed = ["Supplier_ID", "Supplier_Lead_Time_Days", "Reorder_Point", "Unit_Cost", "Unit_Price"]
print(df.groupby(["SKU_ID", "Warehouse_ID"])[fixed].nunique().max())

Supplier_ID                1
Supplier_Lead_Time_Days    1
Reorder_Point              1
Unit_Cost                  1
Unit_Price                 1
dtype: int64


In [7]:
# Does SKU_ID alone fix the supplier? (expect more than 1)
#supplier depends on product and warehouse together, but not on product alone
print(df.groupby("SKU_ID")["Supplier_ID"].nunique().max())

# Does Supplier_ID alone fix the lead time? (expect more than 1)
#supplier_id for different products may have different lead times, so supplier_id alone does not fix the lead time
print(df.groupby("Supplier_ID")["Supplier_Lead_Time_Days"].nunique().max())

5
13
